In [ ]:
!pip install  datasets evaluate transformers[sentencepiece]

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
import evaluate
import numpy as np

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/sst2")

In [ ]:
raw_datasets

In [ ]:
checkpoint = "google-bert/bert-base-uncased"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def tokenize_function(example):
  return tokenizer(example['sentence'],padding=False,truncation=True)

In [ ]:
tokenized_dataset = raw_datasets.map(tokenize_function,batched=True)

In [ ]:
datacollator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
metric = evaluate.load("accuracy")

In [ ]:
from transformers import TrainingArguments

In [ ]:
def compute_metrics(eval_preds):
  logits,labels=eval_preds
  predictions = np.argmax(logits,axis=-1)
  return metric.compute(predictions=predictions,references=labels)

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./bert-sst2-checkpoints",
    per_device_train_batch_size=16, # During training, one GPU processes 16 samples at one time.
    per_device_eval_batch_size=16, # During evaluation , one GPU processes 16 samples at one time.
    num_train_epochs=3,
    max_steps=-1, # steps dominate epochs , here ignore max steps , do till total steps by epochs , total steps = number of batches * number of epochs
    learning_rate =2e-5,
    lr_scheduler_type ="linear",
    optim="adamw_torch",
    gradient_accumulation_steps=2,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end =True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_only_model=True,
    save_total_limit=True,
    logging_dir="./logs",
    logging_steps=50,
    logging_strategy="steps",
    do_eval= True,
    warmup_ratio=0.1,
    weight_decay=0.01,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    report_to="tensorboard",
    seed=42,






)

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_arguments,
    data_collator=datacollator,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,

)

In [ ]:
trainer.train()

In [ ]:
raw_datasets

In [ ]:
# lets look the dataset
print(f"{raw_datasets['train'][1]['sentence']} label={raw_datasets['train'][1]['label']}" )


In [ ]:
from transformers import pipeline
base_classifier = pipeline("text-classification",
                           model="google-bert/bert-base-uncased",
                           tokenizer="google-bert/bert-base-uncased")
print(
    base_classifier("this movie was absolutely fantastic")
)

In [ ]:
print(
    base_classifier("contains no wit , only labored gags")
)

In [ ]:

from transformers import pipeline

fine_tuned_classifier = pipeline(
    "text-classification",
    model="./bert-sst2-checkpoints/checkpoint-2105",
    tokenizer="./bert-sst2-checkpoints/checkpoint-2105"
)

print(
    f"This movie was absolutely fantastic! -> "
    f"{fine_tuned_classifier('This movie was absolutely fantastic!')}"
)

print(
    f"This was the worst product I have ever bought. -> "
    f"{fine_tuned_classifier('This was the worst product I have ever bought.')}"
)



In [ ]:
print(
    f"I love this phone -> "
    f"{fine_tuned_classifier('I love this phone')}"
)

In [ ]:
## model already trained , but we want model to show negative for 0 and 1 for positive  so ,
model.config.id2label = { 0: "NEGATIVE", 1: "POSITIVE" }
model.config.label2id = { "NEGATIVE": 0, "POSITIVE": 1 }

In [ ]:
trainer.save_model("./final-sst2-model")

In [ ]:

from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./final-sst2-model",
    tokenizer="./final-sst2-model"
)




In [ ]:
result = classifier( "This AI project is absolutely amazing!" )
print(result)

In [ ]:

sentences = [

    "I love this movie",

    "This was the worst experience ever",

    "The product quality is amazing",

    "I regret buying this phone",

    "The movie was okay"
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):

    print(f"Sentence: {sentence}")

    print(f"Prediction: {result}")

    print("-" * 50)



In [ ]:
sentences = [

    "The movie was not bad",

    "I expected much better",

    "It was decent but too long",

    "The acting was terrible but visuals were amazing",

    "I don't think I disliked it",

    "The product is fine",

    "The ending ruined the entire film",

    "Not great, not terrible",

    "This is probably one of the movies ever made",

    "It had potential but failed badly"
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):

    print(f"Sentence: {sentence}")

    print(f"Prediction: {result}")

    print("-" * 50)


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub(
    "bert-base-uncased-sst2-finetuned"
)

tokenizer.push_to_hub(
    "bert-base-uncased-sst2-finetuned"
)